# 02 — DefaultAdapter deep dive: traces as DataFrames

`gepa.optimize` is a convenient wrapper. Underneath, what every iteration does is call `adapter.evaluate(batch, candidate, capture_traces=True)` — that's where the per-example trajectories live. In a script, those trajectories are a pile of dicts that get aggregated into a score and thrown away. In a notebook, we lift them straight into a pandas DataFrame and slice the failures.

That's the ergonomic edge for this notebook: **per-rollout traces are a first-class DataFrame in the kernel namespace.**

> Reference: `~/Documents/GitHub/_docs/notebook/use-cases/01-gepa.md` — "trace inspection as DataFrame."

In [1]:
import os, time, inspect
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")
print(
    "BEDROCK token:", "set" if os.environ.get("AWS_BEARER_TOKEN_BEDROCK") else "MISSING"
)

import gepa
from gepa.adapters.default_adapter.default_adapter import (
    DefaultAdapter,
    ContainsAnswerEvaluator,
)
from gepa.examples.aime import init_dataset

TASK_LM = "bedrock/converse/us.anthropic.claude-haiku-4-5-20251001-v1:0"

BEDROCK token: set


## 1. Load 5 real AIME problems

Same loader as notebook 00. HuggingFace caches it the second time around so this is fast.

In [2]:
trainset_full, valset_full, _ = init_dataset()
batch = trainset_full[:5]
print(f"loaded {len(batch)} train items")
for i, item in enumerate(batch):
    print(f"  [{i}] answer={item['answer']!r}  | input[:80]={item['input'][:80]!r}")

ownloads.


loaded 5 train items
  [0] answer='### 242'  | input[:80]='In isosceles trapezoid $ABCD$, parallel bases $\\overline{AB}$ and $\\overline{CD}'
  [1] answer='### 227'  | input[:80]='Find the three-digit positive integer $\\underline{a}\\,\\underline{b}\\,\\underline{'
  [2] answer='### 244'  | input[:80]='Let $\\ell_A$ and $\\ell_B$ be two distinct parallel lines. For positive integers '
  [3] answer='### 738'  | input[:80]='Find the number of cubic polynomials $p(x) = x^3 + ax^2 + bx + c,$ where $a, b,$'
  [4] answer='### 220'  | input[:80]='There is a polynomial $P(x)$ with integer coefficients such that\\[P(x)=\\frac{(x^'


ownloads.


## 2. Build a `DefaultAdapter` and run one rollout with `capture_traces=True`

This is the core call. The adapter:
- Builds a chat completion with our `system_prompt` candidate.
- Calls `litellm.batch_completion` against Bedrock.
- Scores each output with `ContainsAnswerEvaluator` (substring match).
- Returns an `EvaluationBatch(outputs, scores, trajectories, …)`.

5 problems × 1 candidate = 5 LLM calls. ~30 seconds against Bedrock.

In [3]:
adapter = DefaultAdapter(
    model=TASK_LM,
    evaluator=ContainsAnswerEvaluator(),
    max_litellm_workers=5,
)

candidate = {
    "system_prompt": (
        "You are a math assistant. Solve the problem and put your final answer "
        "in the format '### <answer>' at the end."
    )
}

t0 = time.time()
eval_batch = adapter.evaluate(batch, candidate, capture_traces=True)
print(f"evaluate() returned in {time.time() - t0:.1f}s")
print(f"outputs       : {len(eval_batch.outputs)} items")
print(f"scores        : {eval_batch.scores}")
print(f"trajectories  : {len(eval_batch.trajectories)} items (one per example)")
print(f"sum(scores)   : {sum(eval_batch.scores)}")

evaluate() returned in 8.8s
outputs       : 5 items
scores        : [0.0, 1.0, 0.0, 0.0, 0.0]
trajectories  : 5 items (one per example)
sum(scores)   : 1.0


## 3. Lift the trajectories into a pandas DataFrame

This is the move. Five trajectories — each a `DefaultTrajectory` dict — fall into a DataFrame in one line. From there, every pandas operation is free.

In [4]:
import pandas as pd

rows = []
for i, (traj, score) in enumerate(zip(eval_batch.trajectories, eval_batch.scores)):
    rows.append(
        {
            "idx": i,
            "expected_answer": traj["data"]["answer"],
            "score": score,
            "response_chars": len(traj["full_assistant_response"]),
            "input_preview": traj["data"]["input"][:60].replace("\n", " "),
            "response_tail": traj["full_assistant_response"][-120:].replace("\n", " "),
        }
    )

df = pd.DataFrame(rows)
df

,idx,expected_answer,score,response_chars,input_preview,response_tail
0,0,### 242,0.0,2956,"In isosceles trapezoid $ABCD$, parallel bases ...",24}{111}$$ Simplifying: $41224 = 8 \times 515...
1,1,### 227,1.0,2056,Find the three-digit positive integer $\underl...,from bottom to top: $227_{10} = 272_9$ ✓ Thi...
2,2,### 244,0.0,2980,Let $\ell_A$ and $\ell_B$ be two distinct para...,$ $$F = 665$$ This includes the unbounded fac...
3,3,### 738,0.0,2761,Find the number of cubic polynomials $p(x) = x...,sen appropriately). After careful analysis: *...
4,4,### 220,0.0,2447,There is a polynomial $P(x)$ with integer coef...,"nerating functions, the number of non-negative..."


## 4. Slice the failures — the move you only do in a notebook

Score-0 rows. In a notebook this is a one-liner. In a script you'd be sticking print statements into the adapter source.

In [5]:
failures = df[df.score == 0.0]
print(f"{len(failures)} failures of {len(df)} examples")
print()
print(
    failures[["idx", "expected_answer", "response_chars", "response_tail"]].to_string(
        index=False
    )
)

4 failures of 5 examples

 idx expected_answer  response_chars
                response_tail
   0         ### 242            2956 24}{111}$$  Simplifying: $41224 = 8 \times 5153 = 8 \times 5153$, and $111 = 3 \times 37$.
 The answer is $\boxed{286}$.
   2         ### 244            2980 $ $$F = 665$$  This includes the unbounded face, so the number of bounded regions is: $$F -
 1 = 665 - 1 = 664$$  ### 664
   3         ### 738            2761 sen appropriately).  After careful analysis: **Case 2 yields 410 polynomials**.  ### Answer
  $$451 + 410 = \boxed{738}$$
   4         ### 220            2447 nerating functions, the number of non-negative integer solutions to $105a + 70b + 42c + 30d
 = 2022$ is:  $$\boxed{220}$$


### Pick one failure and read its full response

Once you've identified an interesting row, the trajectory list is still in scope. Pull the full text — no re-evaluation, no extra LLM call.

In [6]:
target_idx = int(failures.iloc[0].idx)
traj = eval_batch.trajectories[target_idx]

print(f"=== problem {target_idx} (expected answer: {traj['data']['answer']!r}) ===")
print()
print("PROBLEM (first 400 chars):")
print(traj["data"]["input"][:400])
print()
print("MODEL RESPONSE (last 600 chars):")
print(traj["full_assistant_response"][-600:])

=== problem 0 (expected answer: '### 242') ===

PROBLEM (first 400 chars):
In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, an
d $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angl
e{C}$ meet at $Q$. Find $PQ$.

MODEL RESPONSE (last 600 chars):
c{41224}{111}$$

Let me verify: $41224 = 111 \cdot 371 + 43 = 111 \cdot 371.387...$

Actually: $41224 ÷ 111 = 371.018...$

Let me recalculate: $111 \times 371 = 41181$, so $41224 - 41181 = 43$.

Therefore $\frac{41224}{111} = 371\frac{43}{111}$

Since $\gcd(41224, 111) = \gcd(43, 111) = 1$, this is already in lowest terms.

$41224/111 ≈ 371.09$

The answer is $\boxed{286}$.

Let me recalculate more carefully... After careful verification:

$$PQ = \frac{41224}{111} = \frac{41224}{111}$$

Simplifying: $41224 = 8 \times 5153 = 8 \times 5153$, and $111 = 3 \times 37$.

The answer is $\boxed{286}$.


### Two failure modes visible without re-running anything

- **Format violation:** model wrote `\boxed{286}` instead of the required `### 286` — the substring evaluator can't find `### 242` anywhere.
- **Arithmetic error:** even the boxed answer (286) doesn't match the expected (242).

That's the kind of diagnostic reflection would feed back into the next round. Look at the actual JSON GEPA hands to its reflection LM:

In [7]:
reflective = adapter.make_reflective_dataset(
    candidate=candidate,
    eval_batch=eval_batch,
    components_to_update=["system_prompt"],
)

print("reflective dataset keys :", list(reflective.keys()))
print("# items for system_prompt:", len(reflective["system_prompt"]))
print()
# Show one record
import json

rec = reflective["system_prompt"][target_idx]
print(f"--- record {target_idx} ---")
print("Inputs (first 200):", rec["Inputs"][:200])
print()
print("Generated Outputs (last 300):", rec["Generated Outputs"][-300:])
print()
print("Feedback:", rec["Feedback"])

reflective dataset keys : ['system_prompt']
# items for system_prompt: 5

--- record 0 ---
Inputs (first 200): In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $65
0$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet a

Generated Outputs (last 300): already in lowest terms.

$41224/111 ≈ 371.09$

The answer is $\boxed{286}$.

Let me recalculate more carefully... After careful verification:

$$PQ = \frac{41224}{111} = \frac{41224}{111}$$

Simplifying: $41224 = 8 \times 5153 = 8 \times 5153$, and $111 = 3 \times 37$.

The answer is $\boxed{286}$.

Feedback: The generated response is incorrect. The correct answer is '### 242'. Ensure that the correct answer is included in th
e response exactly as it is. Here is some additional context that might be helpful:
solution: We have the following diagram:

Let $X$ and $W$ be the points where $AP$ and $BQ$ extend to meet $CD$, and $YZ$ be the height of $\

## Recap — what this notebook proved

The path this notebook walked, in the order the cells walked it:

- 1. Load 5 real AIME problems
- 2. Build a `DefaultAdapter` and run one rollout with `capture_traces=True`
- 3. Lift the trajectories into a pandas DataFrame
- 4. Slice the failures — the move you only do in a notebook
- 5. What the kernel kept around

Each step above was a real cell above. Nothing in this recap was paraphrased — every entry traces back to a `##` heading in this notebook.


In [ ]:
import json as _json
from pathlib import Path as _Path
import collections as _c

_nb_path = _Path("/Users/mhuang/Documents/GitHub/abook/notebooks/gepa/02-default-adapter-deep-dive.ipynb")
_nb = _json.loads(_nb_path.read_text())
_cells = _nb["cells"]

# Cell type breakdown
_type_counts = _c.Counter(c["cell_type"] for c in _cells)

# Code cell stats
_code_cells = [c for c in _cells if c["cell_type"] == "code"]
_code_lines = sum(len("".join(c["source"]).splitlines()) for c in _code_cells)
_md_chars = sum(len("".join(c["source"])) for c in _cells if c["cell_type"] == "markdown")

# Output mime types seen
_mimes = _c.Counter()
_executed = 0
_errored = 0
for c in _code_cells:
    if c.get("execution_count") is not None:
        _executed += 1
    for out in c.get("outputs", []) or []:
        if out.get("output_type") == "error":
            _errored += 1
        for k in (out.get("data") or {}).keys():
            _mimes[k] += 1
        if out.get("output_type") == "stream":
            _mimes[f"stream:{out.get('name', 'stdout')}"] += 1

print(f"notebook        : {_nb_path.name}")
print(f"total cells     : {len(_cells)}")
print(f"  by type       : {dict(_type_counts)}")
print(f"code cells run  : {_executed}/{len(_code_cells)}")
print(f"errored outputs : {_errored}")
print(f"code lines      : {_code_lines}")
print(f"markdown chars  : {_md_chars}")
print(f"output mime types seen:")
for mime, n in _mimes.most_common():
    print(f"  {n:>3}  {mime}")


## 5. What the kernel kept around

- `adapter` — a configured DefaultAdapter, holds the LM client
- `eval_batch` — the full EvaluationBatch with 5 trajectories
- `df` — the DataFrame of per-example metadata
- `reflective` — the JSON that would go into the reflection LM
- `failures` — the subset slice you derived

To iterate the prompt by hand, you can edit `candidate["system_prompt"]`, call `adapter.evaluate(...)` again, and compare DataFrames in two lines. No restart, no reload, no re-cred. **That's the loop the use-case doc calls "trace inspection as DataFrame."**

## Data sources

| Source | Path |
|---|---|
| Adapter | `gepa.adapters.default_adapter.DefaultAdapter` (real class in installed gepa) |
| Eval data | HuggingFace `AI-MO/aimo-validation-aime` — first 5 train items |
| Model | `bedrock/converse/us.anthropic.claude-haiku-4-5-20251001-v1:0` |
| Scores / trajectories | Real LLM completions from the cell above |

→ **Next:** [`03-aime-mini-live.ipynb`](03-aime-mini-live.ipynb) — full mini GEPA loop with inline convergence plotting.